# Poisson clipping analysis (noise runs, h512)

Compare Poisson rate/count statistics when mapping hidden activity to spikes **with** vs **without** clipping rates to `[0, rate_max]`.

Uses `area-A0` activity from `runs/noise_runs` (512 neurons per timestep).

In [ ]:
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullLocator

DT = 0.01
RATE_MAX = 40.0
SEED = 42
AREA_KEY = "area-A0"

noise_runs_dir = Path("..") / "runs" / "noise_runs"
noise_run_dirs = sorted(
    [p for p in noise_runs_dir.iterdir() if p.is_dir()],
    key=lambda p: float(re.search(r"_n([\d.]+)_", p.name).group(1)),
)


def parse_noise(run_name: str) -> float:
    return float(re.search(r"_n([\d.]+)_", run_name).group(1))


def activity_to_rates(activity: np.ndarray, *, clip_rates: bool) -> np.ndarray:
    rates = RATE_MAX * (activity + 1.0) / 2.0
    if clip_rates:
        return np.clip(rates, 0.0, RATE_MAX)
    return np.clip(rates, 0.0, None)


def activity_to_lam(activity: np.ndarray, *, clip_rates: bool) -> np.ndarray:
    return activity_to_rates(activity, clip_rates=clip_rates) * DT


def activity_to_counts(activity: np.ndarray, *, clip_rates: bool, rng: np.random.Generator) -> np.ndarray:
    return rng.poisson(activity_to_lam(activity, clip_rates=clip_rates)).astype(np.float32)

In [ ]:
rows = []
rng = np.random.default_rng(SEED)

for run_dir in noise_run_dirs:
    noise = parse_noise(run_dir.name)
    with h5py.File(run_dir / "data.h5", "r") as h:
        activity = h["0"][AREA_KEY][:]

    flat = activity.reshape(-1)
    rates_unclip = activity_to_rates(flat, clip_rates=False)
    rates_clip = activity_to_rates(flat, clip_rates=True)
    lam_unclip = rates_unclip * DT
    lam_clip = rates_clip * DT
    counts_unclip = rng.poisson(lam_unclip).astype(np.float32)
    counts_clip = rng.poisson(lam_clip).astype(np.float32)

    rows.append(
        {
            "noise": noise,
            "run_name": run_dir.name,
            "n_neurons": activity.shape[-1],
            "activity_min": float(flat.min()),
            "activity_max": float(flat.max()),
            "frac_below_-1": float((flat < -1).mean()),
            "frac_above_1": float((flat > 1).mean()),
            "frac_outside_-1_1": float((np.abs(flat) > 1).mean()),
            "frac_rate_capped": float((rates_unclip > RATE_MAX).mean()),
            "mean_lam_unclip": float(lam_unclip.mean()),
            "mean_lam_clip": float(lam_clip.mean()),
            "mean_lam_ratio": float(lam_clip.mean() / lam_unclip.mean()),
            "max_lam_unclip": float(lam_unclip.max()),
            "max_lam_clip": float(lam_clip.max()),
            "mean_count_unclip": float(counts_unclip.mean()),
            "mean_count_clip": float(counts_clip.mean()),
            "mean_count_ratio": float(counts_clip.mean() / counts_unclip.mean()),
            "std_count_unclip": float(counts_unclip.std()),
            "std_count_clip": float(counts_clip.std()),
        }
    )

summary_df = pd.DataFrame(rows).sort_values("noise")
summary_df

In [ ]:
display_cols = [
    "noise",
    "activity_min",
    "activity_max",
    "frac_outside_-1_1",
    "frac_rate_capped",
    "mean_lam_unclip",
    "mean_lam_clip",
    "mean_lam_ratio",
    "max_lam_unclip",
    "max_lam_clip",
    "mean_count_unclip",
    "mean_count_clip",
    "mean_count_ratio",
]
summary_df[display_cols].round(4)

In [ ]:
noise_levels = summary_df["noise"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
ax.plot(noise_levels, summary_df["mean_lam_unclip"], "o-", label="unclipped")
ax.plot(noise_levels, summary_df["mean_lam_clip"], "o-", label="clipped")
ax.set_xscale("log")
ax.set_ylabel("mean λ")
ax.set_title("Mean Poisson rate (λ = rate × dt)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(noise_levels, summary_df["max_lam_unclip"], "o-", label="unclipped")
ax.plot(noise_levels, summary_df["max_lam_clip"], "o-", label="clipped")
ax.set_xscale("log")
ax.set_ylabel("max λ")
ax.set_title("Max λ per run")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(noise_levels, summary_df["frac_outside_-1_1"], "o-", color="C0", label="|activity| > 1")
ax.plot(noise_levels, summary_df["frac_rate_capped"], "o-", color="C1", label="rate > rate_max")
ax.set_xscale("log")
ax.set_ylabel("fraction")
ax.set_title("Activity/rate saturation")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(noise_levels, summary_df["mean_count_unclip"], "o-", label="unclipped")
ax.plot(noise_levels, summary_df["mean_count_clip"], "o-", label="clipped")
ax.set_xscale("log")
ax.set_ylabel("mean count")
ax.set_title(f"Mean Poisson count (dt={DT}, rate_max={RATE_MAX}, seed={SEED})")
ax.legend()
ax.grid(True, alpha=0.3)

for ax in axes.flat:
    ax.xaxis.set_major_locator(FixedLocator(noise_levels))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticklabels([str(n) for n in noise_levels])
    if ax in (axes[1, 0], axes[1, 1]):
        ax.set_xlabel("noise")

fig.suptitle(f"Clipped vs unclipped Poisson mapping ({AREA_KEY}, 512 neurons)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Example λ distributions at low vs high noise
example_noises = [0.05, 5.0]
example_runs = {parse_noise(p.name): p for p in noise_run_dirs}

fig, axes = plt.subplots(len(example_noises), 2, figsize=(12, 8), sharex=True)

for row, noise in enumerate(example_noises):
    with h5py.File(example_runs[noise] / "data.h5", "r") as h:
        activity = h["0"][AREA_KEY][:].reshape(-1)

    lam_unclip = activity_to_lam(activity, clip_rates=False)
    lam_clip = activity_to_lam(activity, clip_rates=True)

    bins = np.linspace(0, max(lam_unclip.max(), lam_clip.max()), 60)

    axes[row, 0].hist(lam_unclip, bins=bins, alpha=0.8, color="C0")
    axes[row, 0].set_ylabel("count")
    axes[row, 0].set_title(f"noise={noise}: λ unclipped")
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].hist(lam_clip, bins=bins, alpha=0.8, color="C1")
    axes[row, 1].set_title(f"noise={noise}: λ clipped")
    axes[row, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("λ")
axes[-1, 1].set_xlabel("λ")
fig.suptitle("λ distributions (all trials × time × neurons)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Single-neuron example: activity → rate mapping curve
activity_grid = np.linspace(-5, 20, 400)
rates_unclip = activity_to_rates(activity_grid, clip_rates=False)
rates_clip = activity_to_rates(activity_grid, clip_rates=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(activity_grid, rates_unclip, label="unclipped")
ax.plot(activity_grid, rates_clip, label="clipped")
ax.axvline(-1, color="gray", linestyle="--", alpha=0.5, label="tanh bounds")
ax.axvline(1, color="gray", linestyle="--", alpha=0.5)
ax.axhline(RATE_MAX, color="gray", linestyle=":", alpha=0.5, label="rate_max")
ax.set_xlabel("activity")
ax.set_ylabel("rate (Hz)")
ax.set_title(f"Mapping activity → rate (rate_max={RATE_MAX})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()